# Antibody Aggregation Failure Pipeline
Run cells top to bottom.

In [ ]:
!git clone https://github.com/saptarshighosh10-oss/Biostuff.git
%cd Biostuff
!git checkout claude/antibody-aggregation-pipeline-xdx1s0
!pip install requests fair-esm -q

In [ ]:
# Basic run — BLOSUM62 mutations, no ESMFold (fast, ~3 min)
!python pipeline.py --entries 300 --variants 5 --mutations 2

In [ ]:
# Big run — 2000 PDB entries (antibodies + nanobodies + VHH + scFv + bispecifics)
# 20 variants each, ESM-2 guided, ESMFold on top 200 (~50-60 min)
# Parallel fetching keeps the network step to ~3 min
!python pipeline.py --entries 2000 --variants 20 --mutations 2 --esm2 --esmfold --top 200

In [ ]:
import json
import collections

with open('results/phase1_candidates.json') as f:
    candidates = json.load(f)

print(f'Total candidates saved: {len(candidates)}')

# --- Risk score distribution ---
risks = [c['risk']['combined_risk'] for c in candidates]
buckets = [('0.6+', 0.6, 1.1), ('0.5-0.6', 0.5, 0.6), ('0.4-0.5', 0.4, 0.5), ('<0.4', 0.0, 0.4)]
print('\nRisk score distribution:')
for label, lo, hi in buckets:
    n = sum(1 for r in risks if lo <= r < hi)
    print(f'  {label:8s} {"\u2588" * n} ({n})')

# --- Chain type breakdown ---
print('\nChain type breakdown:')
for t, n in collections.Counter(c['chain_type'] for c in candidates).most_common():
    print(f'  {t:8s}: {n}')

# --- Mutation hotspots ---
print('\nTop 10 mutated positions across all candidates:')
pos_counts = collections.Counter()
mut_counts = collections.Counter()
for c in candidates:
    for pos, orig, mut in c['mutations']:
        pos_counts[pos] += 1
        mut_counts[f'{orig}\u2192{mut}'] += 1
for pos, count in pos_counts.most_common(10):
    print(f'  position {pos:4d}: seen in {count} candidates')

print('\nTop 10 substitution types:')
for sub, count in mut_counts.most_common(10):
    print(f'  {sub}: {count}')

# --- Top 5 detailed ---
print('\n' + '='*60)
print('TOP 5 CANDIDATES (detailed)')
print('='*60)
for i, c in enumerate(candidates[:5]):
    r = c['risk']
    dscore = r.get('disagreement_score')
    dscore_str = f' | disagreement={dscore:.3f}' if dscore is not None else ''
    struct = r.get('structure_features', {})
    plddt_str = ''
    if struct:
        plddt_str = f' | mean_pLDDT={struct.get("mean_plddt", 0):.1f} | low_conf={struct.get("low_confidence_fraction", 0):.2f}'
    print(f'\n{i+1}. {c["anchor_pdb"]} ({c["chain_type"]})')
    print(f'   mutations:    {c["mutations"]}')
    print(f'   risk:         {r["combined_risk"]:.3f}{dscore_str}')
    print(f'   CamSol:       {r.get("camsol_score", 0):.3f}  (lower = worse solubility)')
    print(f'   hydrophobic:  {r.get("hydrophobicity_score", 0):.3f}{plddt_str}')


In [ ]:
from google.colab import files
files.download('results/phase1_candidates.json')

In [ ]:
# Phase 2 — multi-predictor consensus + explanation (~5 min, no MD)
!pip install openmm -q
!python pipeline_phase2.py --top 20

In [ ]:
# Phase 2 with MD — adds per-residue flexibility data (~30 min, top 10 only)
# !python pipeline_phase2.py --top 10 --md

In [ ]:
# Explore Phase 2 results
import json
with open('results/phase2_candidates.json') as f:
    candidates = json.load(f)

print(f'Candidates with explanations: {len(candidates)}')
for i, c in enumerate(candidates[:5]):
    mp = c.get('multi_predictor', {})
    hotspots = mp.get('consensus_hotspots', [])
    print(f"\n{i+1}. {c['anchor_pdb']} ({c['chain_type']}) | risk={c['risk']['combined_risk']:.3f}")
    print(f"   mutations:  {c['mutations']}")
    print(f"   hotspots:   {hotspots if hotspots else 'none'}")
    print(f"   explanation: {c['explanation']}")

In [ ]:
# Phase 3 — database cross-referencing (~5 min for top 50)
# Queries EBI PDBe, UniProt REST, EBI Proteins API — all free, no key needed
!python pipeline_phase3.py --top 50

In [ ]:
# Explore Phase 3 results
import json
with open('results/phase3_candidates.json') as f:
    candidates = json.load(f)

print(f'Candidates with database evidence: {len(candidates)}\n')

from collections import Counter
conf_counts = Counter(c['database_evidence']['database_confidence'] for c in candidates)
print('Database confidence breakdown:')
for level in ['high', 'medium', 'low', 'none']:
    n = conf_counts.get(level, 0)
    print(f'  {level:6s}: {"█" * n} ({n})')

print('\nTop 5 after database re-ranking:')
for i, c in enumerate(candidates[:5]):
    ev = c['database_evidence']
    print(f"\n{i+1}. {c['anchor_pdb']} ({c.get('chain_type','?')}) | boosted_risk={c['risk']['db_boosted_risk']:.3f}")
    print(f"   mutations:  {c['mutations']}")
    print(f"   db level:   {ev['database_confidence']}")
    if ev.get('protein_name'):
        print(f"   protein:    {ev['protein_name']}")
    print(f"   evidence:   {c['db_explanation'][:150]}")

In [ ]:
# Train model — MAX DATA run (no wet lab needed)
#   FLAb       — experimental aggregation assays → failures + working
#   AbDev      — clinical-stage antibodies → clean negatives
#   anchors    — your Phase 1 PDB anchor chains → free negatives
#   proteingym — ALL 84 stability/expression DMS assays (~thousands of mutant
#                failures incl. a 14,811-variant amyloid-beta aggregation scan).
#                Seeded random sampling (seed 42) spreads coverage across the
#                whole fitness range; these mutants activate the delta features.
#   SAbDab     — structural antibody negatives (skipped gracefully if blocked)
# This downloads ~84 CSVs — give it a few minutes.
!pip install scikit-learn joblib -q
!python -m model.train --flab --abdev --anchors --proteingym --sabdab --flab-percentile 0.4

# Lighter run (fewer assays, faster):
# !python -m model.train --flab --abdev --anchors --proteingym --proteingym-assays 15 --flab-percentile 0.4

In [ ]:
# Optional robustness check — retrain with 3 different seeds and compare AUC.
# If AUC is stable across seeds, the model is learning real signal, not noise.
# for s in 42 43 44:
#     print(f'=== seed {s} ===')
#     !python -m model.train --flab --abdev --anchors --proteingym --proteingym-seed $s --flab-percentile 0.4 2>&1 | grep -E "Best AUC|Failures:"

In [ ]:
# Re-score your Phase 3 candidates using the trained model
!python -m model.predict --file results/phase3_candidates.json --top 20

In [ ]:
# Rescue-mutation suggester — for your top candidate, find the smallest change
# that most reduces predicted risk (revert to wild-type, or a better substitution).
# Also reports the closest known WORKING and FAILURE reference proteins
# ("this candidate most resembles known-working protein X").
!python -m model.rescue --file results/phase3_candidates.json --index 0 --top 5

# Try a different candidate by changing --index (0 = highest risk)

In [ ]:
# Demo validation — test the trained model on textbook extreme cases
# it NEVER saw: famous aggregators (Abeta, alpha-synuclein, IAPP, prion...)
# vs famous soluble proteins (ubiquitin, GFP, lysozyme, cytochrome c...).
# Shows whether the antibody-trained features capture general aggregation.
!python -m model.validate --plot --save results/benchmark_scored.json

from IPython.display import Image
display(Image('results/viz/benchmark_validation.png'))

## Active Learning Loop
After wet-lab testing: fill in `results/wetlab_results.json`, run feedback, retrain, repeat.

In [ ]:
# Step 1 — See what to test next (ranked by risk × uncertainty)
# Run this before going to the lab to know which candidates are worth your time
!python -m model.feedback --priority results/phase3_candidates.json --top 10

In [ ]:
# Step 2 — After testing: fill in your results, then run this to retrain
# Copy results/wetlab_results_template.json → results/wetlab_results.json
# Fill in variant_sequence, label ("confirmed_failure" or "working"), and notes
# Then run:
!python -m model.feedback --results results/wetlab_results.json

In [ ]:
# Step 3 — Re-score all candidates with the updated model
# Candidates that were uncertain before may now rank differently
!python -m model.predict --file results/phase3_candidates.json --top 20

In [ ]:
# Visualize results — works with phase1, phase2, phase3, or model-scored candidates
# Auto-detects the best available file
!python visualize.py

from IPython.display import Image
import glob, os
imgs = sorted(glob.glob('results/viz/*.png'), key=os.path.getmtime)
if imgs:
    display(Image(imgs[-1]))